# 답변 근거 선택·생성 검사 보완 결과
[프로젝트 추가] 사용자 확인 후 수정한 결과입니다. 아래 셀은 저장 파일만 읽습니다. DB 검색·임베딩·생성 API를 다시 실행하지 않습니다.
28번 노트북의 이전 평가 결과는 그대로 보존합니다. 먼저 [쉬운 보완 안내](./answer_fix_guide.md)를 참고하세요.

In [ ]:
# [프로젝트 추가] 노트북을 연 위치와 무관하게 프로젝트와 개인 요약을 찾습니다.
from pathlib import Path
import json
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/car_search_rag/zzong_santafe_lag').is_dir())
private_folder = project / 'src/car_search_rag/zzong_santafe_lag'
summary = json.loads((private_folder / 'answer_fix_summary_20261005.json').read_text(encoding='utf-8'))
folder = project / summary['details_folder']
print('결과 폴더:', folder)
print('기존 조건 일치:', summary['regression_expectation_match'])
print('인용 규칙 확인:', summary['policy_checks_passed'])
print('이번 보완 생성 사용량:', summary['usage'])


In [ ]:
# [프로젝트 추가] 검색 순위는 그대로 두고 답변에 선택한 자료가 달라진 문항만 확인합니다.
selection = json.loads((folder / summary['selection_report']).read_text(encoding='utf-8'))
for row in selection['results']:
    before = [s['title'] for s in row['before']['sources']]
    after = [s['title'] for s in row['after']['sources']]
    if before != after:
        print(row['backend'], row['case_id'], before, '→', after)
print('검색 Precision@5는 기존 잠정 36% 그대로입니다. 이것은 근거 선택의 비교입니다.')


In [ ]:
# [프로젝트 추가] 이번 진단에서 실제로 확인한 오류 코드와 모델 글을 읽습니다.
diagnosis = json.loads((folder / summary['diagnosis_report']).read_text(encoding='utf-8'))
for row in diagnosis['results']:
    diagnostic = row['answer'].get('generation_diagnostics', {})
    print('\n문항:', row['case_id'], '| 검사:', diagnostic.get('code', '통과'))
    print('누락 출처:', diagnostic.get('missing_citation_ids', []))
    print('모델 원문:', diagnostic.get('raw_response', '없음'))


In [ ]:
# [프로젝트 추가] 최종 코드로 한 번씩 생성한 답변입니다. API를 새로 호출하지 않습니다.
final = json.loads((folder / summary['final_generation_report']).read_text(encoding='utf-8'))
for row in final['results']:
    answer = row['answer']
    print('\n문항:', row['case_id'], '| 상태:', answer['status'])
    print(answer['answer'])
    print('사용 토큰:', answer['token_usage'])
    print('인용 조건:', [(s['title'], s.get('citation_required', True)) for s in answer['sources']])
print('3문항의 생성 확인입니다. 전체 챗봇 정확도 100%를 뜻하지 않습니다.')
